In [3]:
from openai import OpenAI
import base64
from dotenv import load_dotenv
import os
from openai import Client
import json
import time

In [4]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if  api_key:
    print("Api key exist")
else:
    print("api key doesnt exist")
    


Api key exist


In [ ]:

Client=OpenAI(api_key=api_key)

In [17]:
UPLOAD_FOLDER = "uploads"
MEMORY_FILE = "database/master_memory.json"

# Load Previous Memory
with open(MEMORY_FILE, "r") as file:
    memory = json.load(file)

# Already processed images
processed_images = {
    item["card.ong"] for item in memory
}


In [20]:
for image_name in os.listdir(UPLOAD_FOLDER):

    # Skip already processed images
    if image_name in processed_images:
        print(f"⏩ Skipping : {image_name}")
        continue

    image_path = os.path.join(UPLOAD_FOLDER, image_name)

    print(f"\n📄 Processing : {image_name}")

    with open(image_path, "rb") as image_file:
        image_base64 = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    response = Client.responses.create(

        model="gpt-4o-mini",

        input=[
            {
                "role": "user",

                "content": [

                    {
                        "type": "input_text",

                        "text": """
You are an intelligent document extraction assistant.

Extract EVERY piece of information visible in this image.

Return ONLY valid JSON.

Example:

{
    "document_type":"",
    "name":"",
    "designation":"",
    "company":"",
    "phone":[],
    "email":[],
    "website":"",
    "address":"",
    "social_media":[],
    "other_details":"",
    "raw_text":""
}

If any field is missing return null.

Do not return markdown.
Do not return explanation.
Return JSON only.
"""
                    },

                    {
                        "type": "input_image",

                        "image_url":
                        f"data:image/png;base64,{image_base64}"
                    }

                ]
            }
        ]
    )

    # -----------------------
    # Convert GPT Output
    # -----------------------
    try:
        extracted_data = json.loads(response.output_text)

    except Exception:

        extracted_data = {
            "raw_output": response.output_text
        }

    # -----------------------
    # Store Information
    # -----------------------
    image_record = {

        "image_id": len(memory) + 1,

        "image_name": image_name,

        "data": extracted_data

    }

    memory.append(image_record)

    print(response.output_text)


📄 Processing : card.png
{
    "document_type":"business_card",
    "name":"ARTHUR J. WRIGHT",
    "designation":"SENIOR SOFTWARE ARCHITECT",
    "company":"INNOVATECH SOLUTIONS",
    "phone":["+1 (555) 123-4567"],
    "email":["arthur.wright@innovatech.com"],
    "website":"www.innovatechsolutions.co",
    "address":"San Francisco, CA, USA",
    "social_media":["linkedin.com/in/arthurjwright"],
    "other_details":"Specializing in scalable cloud infrastructure and enterprise software solutions. CORE SKILLS: Cloud Architecture (AWS/Azure), Microservices & Kubernetes, Agile Methodology.",
    "raw_text":"ARTHUR J. WRIGHT SENIOR SOFTWARE ARCHITECT INNOVATECH SOLUTIONS +1 (555) 123-4567 arthur.wright@innovatech.com www.innovatechsolutions.co San Francisco, CA, USA Specializing in scalable cloud infrastructure and enterprise software solutions. CORE SKILLS: Cloud Architecture (AWS/Azure) Microservices & Kubernetes Agile Methodology linkedin.com/in/arthurjwright"
}


In [19]:
with open(MEMORY_FILE, "w") as file:
    json.dump(memory, file, indent=4)

print("\n🎉 Finished Processing Images")


🎉 Finished Processing Images
